<a href="https://colab.research.google.com/github/AisyahFadhillah/Sampling-Techniques-and-Data-Wrangling/blob/main/UTS_TPSDW_241061013_AISYAH_FADHILLAH_MUHAMMAD_AZIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
NAMA : AISYAH FADHILLAH MUHAMMAD AZIS

NIM  : 241061013

MATA KULIAH : TEKNIK PENGAMBILAN SAMPLING & DATA WRANGLING

UJIAN TENGAH SEMESTER (UTS)

---


**Instruksi UTS**

Paket 1: NIM akhiran: 0, 3 , 6 dan 7

1.	Lakukan scraping web (https://deepublishstore.com/product-category/ekonomi/page/1/?) di google colab, scraping halaman 1- 4, dengan mengambil informasi
-	Judul Buku (code: ("h2", class_="woocommerce-loop-product__title"))
-	Harga Buku (code: ("span", class_="woocommerce-Price-amount amount"))
-	Link Gambar Buku (code: ("img", class_="attachment-woocommerce_thumbnail size-woocommerce_thumbnail"))
2.	Berikan nama hasil scraping bernama List_Buku, ubah format file menjadi JSON
3.	Upload data json ke github
4.	Ubah format data menjadi csv menggunakan prinsip ETL. Upload hasil ETL dalam bentuk csv ke github



In [19]:
# IMPORT LIBRARY
!pip install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import json

In [28]:
# Kode ini hanya untuk melihat struktur HTML

#url = "https://deepublishstore.com/product-category/ekonomi/page/1/?"
#headers = {"User-Agent": "Mozilla/5.0"}
#response = requests.get(url, headers=headers)
#soup = BeautifulSoup(response.text, "html.parser")
#print(soup.prettify())

In [29]:
# SCRAPPING (1 - 4 HALAMAN)
headers = {
    "User-Agent": "Mozilla/5.0"
}

base_url = "https://deepublishstore.com/product-category/ekonomi/page/"

data_buku = []

for page in range(1, 5):
    url = f"{base_url}{page}/"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    products = soup.find_all("li", class_="product")

    for p in products:
        # Judul
        judul = p.find("h2", class_="woocommerce-loop-product__title")
        judul = judul.text.strip() if judul else "N/A"

        # Harga
        harga = p.find("span", class_="woocommerce-Price-amount amount")
        harga = harga.text.strip() if harga else "N/A"

        # Gambar
        gambar = p.find("img", class_="attachment-woocommerce_thumbnail size-woocommerce_thumbnail")
        gambar = gambar["src"] if gambar else "N/A"

        data_buku.append({
            "judul": judul,
            "harga": harga,
            "gambar": gambar
        })

print(f"Total data berhasil di-scrape: {len(data_buku)}")

Total data berhasil di-scrape: 96


In [30]:
# TAMPILKAN DATA
df = pd.DataFrame(data_buku)
df = df[["judul", "harga", "gambar"]]

print("\nPreview Data:")
display(df.head(10))


Preview Data:


,judul,harga,gambar
0,Buku Pengantar Akuntansi Sektor Publik Ragam ...,Rp 87.000,https://deepublishstore.com/wp-content/uploads...
1,Buku Ekspor Impor: Teori dan Praktikum Kegiat...,Rp 260.000,https://deepublishstore.com/wp-content/uploads...
2,Buku Strategi Pemasaran Retail Jilid 1,Rp 102.000,https://deepublishstore.com/wp-content/uploads...
3,Buku Referensi Statistika untuk Ekonomi dan B...,Rp 83.000,https://deepublishstore.com/wp-content/uploads...
4,Buku Ajar: Sistem Pendukung Keputusan Teori d...,Rp 61.000,https://deepublishstore.com/wp-content/uploads...
5,Buku Pengantar Akuntansi Integrasi Akuntansi ...,Rp 110.000,https://deepublishstore.com/wp-content/uploads...
6,Buku Manajemen Pengadaan Barang dan Jasa Peme...,Rp 143.000,https://deepublishstore.com/wp-content/uploads...
7,Buku Perilaku Organisasi,Rp 94.000,https://deepublishstore.com/wp-content/uploads...
8,Buku Peran Bank Syariah dalam Pemulihan Ekono...,Rp 57.000,https://deepublishstore.com/wp-content/uploads...
9,Buku Ajar Kewirausahaan untuk Mahasiswa Keseh...,Rp 59.000,https://deepublishstore.com/wp-content/uploads...


In [22]:
# SIMPAN KE JSON
with open("List_Buku.json", "w", encoding="utf-8") as f:
    json.dump(data_buku, f, indent=4, ensure_ascii=False)

print("File JSON berhasil dibuat: List_Buku.json")

File JSON berhasil dibuat: List_Buku.json


In [23]:
# ETL PROCESS

# Extract
df_etl = pd.read_json("List_Buku.json")

# Transform (Cleaning Harga)
def clean_price(x):
    if x == "N/A":
        return None
    return int(x.replace("Rp", "").replace(".", "").strip())

df_etl["harga_clean"] = df_etl["harga"].apply(clean_price)

# Transform (Kategori Harga)
def kategori_harga(x):
    if x is None:
        return "Unknown"
    elif x < 50000:
        return "Murah"
    elif x < 100000:
        return "Sedang"
    else:
        return "Mahal"

df_etl["kategori"] = df_etl["harga_clean"].apply(kategori_harga)

In [24]:
# LOAD KE CSV
df_etl.to_csv("hasil_ETL_buku.csv", index=False)

print("File CSV berhasil dibuat: hasil_ETL_buku.csv")

File CSV berhasil dibuat: hasil_ETL_buku.csv


In [25]:
# TAMPILKAN HASIL ETL
print("\nPreview Data ETL:")
display(df_etl.head(10))


Preview Data ETL:


,judul,harga,gambar,harga_clean,kategori
0,Buku Pengantar Akuntansi Sektor Publik Ragam ...,Rp 87.000,https://deepublishstore.com/wp-content/uploads...,87000,Sedang
1,Buku Ekspor Impor: Teori dan Praktikum Kegiat...,Rp 260.000,https://deepublishstore.com/wp-content/uploads...,260000,Mahal
2,Buku Strategi Pemasaran Retail Jilid 1,Rp 102.000,https://deepublishstore.com/wp-content/uploads...,102000,Mahal
3,Buku Referensi Statistika untuk Ekonomi dan B...,Rp 83.000,https://deepublishstore.com/wp-content/uploads...,83000,Sedang
4,Buku Ajar: Sistem Pendukung Keputusan Teori d...,Rp 61.000,https://deepublishstore.com/wp-content/uploads...,61000,Sedang
5,Buku Pengantar Akuntansi Integrasi Akuntansi ...,Rp 110.000,https://deepublishstore.com/wp-content/uploads...,110000,Mahal
6,Buku Manajemen Pengadaan Barang dan Jasa Peme...,Rp 143.000,https://deepublishstore.com/wp-content/uploads...,143000,Mahal
7,Buku Perilaku Organisasi,Rp 94.000,https://deepublishstore.com/wp-content/uploads...,94000,Sedang
8,Buku Peran Bank Syariah dalam Pemulihan Ekono...,Rp 57.000,https://deepublishstore.com/wp-content/uploads...,57000,Sedang
9,Buku Ajar Kewirausahaan untuk Mahasiswa Keseh...,Rp 59.000,https://deepublishstore.com/wp-content/uploads...,59000,Sedang
